In [2]:
#r "nuget: Deedle.Interactive"
#r "nuget: FsHttp"

Installed Packages Deedle.Interactive, 3.0.0 FsHttp, 15.0.3

Loading extensions from `C:\Users\schne\.nuget\packages\deedle.interactive\3.0.0\lib\netstandard2.1\Deedle.Interactive.dll`

In [3]:
open Deedle

let df = Frame.ReadCsv("./ProjectsWithTotalAttributes.csv")
df

0,->,PRJNA852208,10533
1,->,PRJEB15671,8315
2,->,PRJEB25079,6986
3,->,PRJNA945404,6909
4,->,PRJEB19252,4882
:,,...,...
821,->,PRJNA319661,22
822,->,PRJNA718567,18
823,->,PRJNA117657,18
824,->,PRJNA744531,16
825,->,PRJNA79813,15


In [4]:
open FsHttp
open System.Xml.Linq

let getAlias acc = 
    http {
        GET $"https://www.ebi.ac.uk/ena/browser/api/xml/{acc}"
    }
    |> Request.send
    |> Response.toText
    |> XDocument.Parse
    |> fun doc -> doc.Descendants(XName.Get "SECONDARY_ID")
    |> Seq.map (fun e -> e.Value)
    |> String.concat ";"


In [5]:
let mapped_df =
    df
    |> fun f ->
        let alias_col =
            f
            |> Frame.mapRows (fun rk os ->
                let acc = os.GetAs<string>("Project_Accession")
                printfn "getting alias for %s" acc
                getAlias acc
            )
        f
        |> Frame.addCol "Alias" alias_col

getting alias for PRJNA852208
getting alias for PRJEB15671
getting alias for PRJEB25079
getting alias for PRJNA945404
getting alias for PRJEB19252
getting alias for PRJNA236110
getting alias for PRJNA808745
getting alias for PRJNA368916
getting alias for PRJNA358059
getting alias for PRJNA666631
getting alias for PRJNA475542
getting alias for PRJNA817215
getting alias for PRJNA989636
getting alias for PRJNA915364
getting alias for PRJNA504621
getting alias for PRJNA483458
getting alias for PRJNA302221
getting alias for PRJNA770533
getting alias for PRJNA515610
getting alias for PRJNA611770
getting alias for PRJNA638728
getting alias for PRJNA841954
getting alias for PRJNA663790
getting alias for PRJNA900135
getting alias for PRJNA779072
getting alias for PRJNA382136
getting alias for PRJNA615216
getting alias for PRJNA224705
getting alias for PRJNA551415
getting alias for PRJNA378644
getting alias for PRJNA689309
getting alias for PRJNA727531
getting alias for PRJNA871888
getting alias

In [6]:
mapped_df

0,->,PRJNA852208,10533,SRP383392
1,->,PRJEB15671,8315,ERP017503
2,->,PRJEB25079,6986,ERP106963
3,->,PRJNA945404,6909,SRP427656
4,->,PRJEB19252,4882,ERP021242
:,,...,...,...
821,->,PRJNA319661,22,SRP073942
822,->,PRJNA718567,18,SRP312746
823,->,PRJNA117657,18,SRP002566
824,->,PRJNA744531,16,SRP327393
825,->,PRJNA79813,15,SRP004053


In [7]:
open System.Text.RegularExpressions

let dee2DumpUrl org acc =
    let html =
        http {
            GET $"https://dee2.io/cgi-bin/search2.sh?org={org}&accessionsearch={acc}"
        }
        |> Request.send
        |> Response.toText
    if html.Contains "No results found" then
        None
    else
        let m = Regex.Match(html, @"href=(https?://[^\s""'>]+\.zip)")
        if m.Success then Some m.Groups.[1].Value else None

dee2DumpUrl "athaliana" "SRP427656"

Value,https://dee2.io/huge/athaliana/SRP427656_GSE227500.zip


In [8]:
let with_dee2_urls =
    mapped_df
    |> fun f ->
        let dee2_col =
            f
            |> Frame.mapRows (fun rk os ->
                let acc = os.GetAs<string>("Alias")
                printfn "getting dee2 url for %s" acc
                dee2DumpUrl "athaliana" acc
                |> Option.defaultValue ""
            )
        let f' = f |> Frame.addCol "dee2_url" dee2_col

        let has_dee_col =
            f'
            |> Frame.mapRows (fun rk os ->
                let acc = os.GetAs<string>("dee2_url")
                if acc = "" then false else true
            )
        f'
        |> Frame.addCol "has_dee2" has_dee_col

            

getting dee2 url for SRP383392
getting dee2 url for ERP017503
getting dee2 url for ERP106963
getting dee2 url for SRP427656
getting dee2 url for ERP021242
getting dee2 url for SRP035593
getting dee2 url for SRP360684
getting dee2 url for SRP097877
getting dee2 url for SRP095347
getting dee2 url for SRP285902
getting dee2 url for SRP150217
getting dee2 url for SRP364447
getting dee2 url for SRP446862
getting dee2 url for SRP414670
getting dee2 url for SRP168223
getting dee2 url for SRP155742
getting dee2 url for SRP066214
getting dee2 url for SRP340951
getting dee2 url for SRP179923
getting dee2 url for SRP252208
getting dee2 url for SRP266864
getting dee2 url for SRP376838
getting dee2 url for SRP282567
getting dee2 url for SRP407150
getting dee2 url for SRP345289
getting dee2 url for SRP103736
getting dee2 url for SRP254043
getting dee2 url for SRP031910
getting dee2 url for SRP212174
getting dee2 url for SRP101641
getting dee2 url for SRP300093
getting dee2 url for SRP318562
getting 

In [12]:
with_dee2_urls.SaveCsv("./ProjectsWithTotalAttributesWithDee2Urls.csv", includeRowKeys=false)

In [27]:
let candidates = 
    with_dee2_urls
    |> Frame.filterRows (fun rk os ->
        os.GetAs<bool>("has_dee2")
    )
    |> Frame.mapRows (fun rk os ->
        os.GetAs<string>("Project_Accession"), os.GetAs<int>("Total_Attributes")
    )
    |> Series.values
    |> Seq.sortByDescending (fun (_, total) -> total)
    |> Seq.take 39
    |> Array.ofSeq

candidates



index value 0 (PRJEB25079, 6986) Item1 PRJEB25079 Item2 6986 1 (PRJNA945404, 6909) Item1 PRJNA945404 Item2 6909 2 (PRJNA808745, 3479) Item1 PRJNA808745 Item2 3479 3 (PRJNA368916, 2724) Item1 PRJNA368916 Item2 2724 4 (PRJNA358059, 2516) Item1 PRJNA358059 Item2 2516 5 (PRJNA666631, 2503) Item1 PRJNA666631 Item2 2503 6 (PRJNA475542, 2359) Item1 PRJNA475542 Item2 2359 7 (PRJNA989636, 2207) Item1 PRJNA989636 Item2 2207 8 (PRJNA915364, 1995) Item1 PRJNA915364 Item2 1995 9 (PRJNA504621, 1929) Item1 PRJNA504621 Item2 1929 10 (PRJNA483458, 1732) Item1 PRJNA483458 Item2 1732 11 (PRJNA302221, 1672) Item1 PRJNA302221 Item2 1672 12 (PRJNA515610, 1595) Item1 PRJNA515610 Item2 1595 13 (PRJNA611770, 1537) Item1 PRJNA611770 Item2 1537 14 (PRJNA638728, 1517) Item1 PRJNA638728 Item2 1517 15 (PRJNA841954, 1460) Item1 PRJNA841954 Item2 1460 16 (PRJNA663790, 1440) Item1 PRJNA663790 Item2 1440 17 (PRJNA779072, 1429) Item1 PRJNA779072 Item2 1429 18 (PRJNA382136, 1407) Item1 PRJNA382136 Item2 1407 19 (PRJNA224705, 1289) Item1 PRJNA224705 Item2 1289 (19 more)

In [ ]:
// --- Create one empty GitLab repo per candidate accession ---
open FsHttp

let gitlabHost = "https://git.nfdi4plants.org"   // TODO: your instance, no trailing slash
let groupId    = 238457374                                // TODO: numeric group ID (Group -> Settings -> General)
let token      = "NAH BRO"  // PAT with `api` scope

let createRepo (acc: string) =
    let payload =
        sprintf "{\"name\":\"%s\",\"path\":\"%s\",\"namespace_id\":%d,\"visibility\":\"private\"}" acc acc groupId
    let resp =
        http {
            POST $"{gitlabHost}/api/v4/projects"
            header "PRIVATE-TOKEN" token
            body
            json payload
        }
        |> Request.send
    let status = int resp.statusCode
    if status = 201 then
        printfn "created %s" acc
    else
        printfn "FAILED %s -> %d: %s" acc status (resp |> Response.toText)
    acc, status

let results =
    candidates
    |> Array.map (fun (acc, _) -> createRepo acc)

created PRJEB25079
created PRJNA945404
created PRJNA808745
created PRJNA368916
created PRJNA358059
created PRJNA666631
created PRJNA475542
created PRJNA989636
created PRJNA915364
created PRJNA504621
created PRJNA483458
created PRJNA302221
created PRJNA515610
created PRJNA611770
created PRJNA638728
created PRJNA841954
created PRJNA663790
created PRJNA779072
created PRJNA382136
created PRJNA224705
created PRJNA551415
created PRJNA378644
created PRJNA727531
created PRJNA871888
created PRJNA691022
created PRJNA419306
created PRJNA641096
created PRJNA429659
created PRJNA481720
created PRJNA730903
created PRJNA1014093
created PRJNA737113
created PRJNA938512
created PRJNA566089
created PRJNA544417
created PRJNA478998
created PRJNA630258
created PRJNA575019
created PRJNA628644


In [29]:
// --- Web URLs for the candidate repos ---
open System.Text.Json

// fetch the target group's web URL once
let groupWebUrl =
    http {
        GET $"{gitlabHost}/api/v4/groups/{groupId}"
        header "PRIVATE-TOKEN" token
    }
    |> Request.send
    |> Response.toText
    |> JsonDocument.Parse
    |> fun d -> d.RootElement.GetProperty("web_url").GetString()

// path was set to the accession at creation, so URL = group/web_url + "/" + accession
let repoUrls =
    candidates
    |> Array.map (fun (acc, _) -> acc, $"{groupWebUrl}/{acc}")

repoUrls


index value 0 (PRJEB25079, https://git.nfdi4plants.org/groups/insdc_curation/PRJEB25079) Item1 PRJEB25079 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJEB25079 1 (PRJNA945404, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA945404) Item1 PRJNA945404 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA945404 2 (PRJNA808745, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA808745) Item1 PRJNA808745 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA808745 3 (PRJNA368916, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA368916) Item1 PRJNA368916 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA368916 4 (PRJNA358059, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA358059) Item1 PRJNA358059 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA358059 5 (PRJNA666631, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA666631) Item1 PRJNA666631 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA666631 6 (PRJNA475542, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA475542) Item1 PRJNA475542 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA475542 7 (PRJNA989636, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA989636) Item1 PRJNA989636 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA989636 8 (PRJNA915364, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA915364) Item1 PRJNA915364 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA915364 9 (PRJNA504621, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA504621) Item1 PRJNA504621 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA504621 10 (PRJNA483458, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA483458) Item1 PRJNA483458 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA483458 11 (PRJNA302221, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA302221) Item1 PRJNA302221 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA302221 12 (PRJNA515610, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA515610) Item1 PRJNA515610 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA515610 13 (PRJNA611770, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA611770) Item1 PRJNA611770 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA611770 14 (PRJNA638728, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA638728) Item1 PRJNA638728 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA638728 15 (PRJNA841954, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA841954) Item1 PRJNA841954 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA841954 16 (PRJNA663790, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA663790) Item1 PRJNA663790 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA663790 17 (PRJNA779072, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA779072) Item1 PRJNA779072 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA779072 18 (PRJNA382136, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA382136) Item1 PRJNA382136 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA382136 19 (PRJNA224705, https://git.nfdi4plants.org/groups/insdc_curation/PRJNA224705) Item1 PRJNA224705 Item2 https://git.nfdi4plants.org/groups/insdc_curation/PRJNA224705 (19 more)